# Phase 5: Audit when each column is known

Each source column is assigned exactly one bin: 1 fixed at booking, 2 amendable or partly known at booking, or 3 after the fact. This notebook does not train a model, calculate AUC, or create `src/features.py`.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

SRC_DIR = Path.cwd().parent / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
import data

bookings = data.read_raw()
audit_rows = [
    ("hotel", 1, "Selected when reservation is made.", "Dataset hotel field."),
    ("is_canceled", 3, "Outcome being predicted; known after cancellation/no-show handling.", "src/data.py defines the outcome column."),
    ("lead_time", 1, "Days between booking and arrival; known at booking.", "Dataset booking/arrival field."),
    ("arrival_date_year", 1, "Requested arrival year.", "Dataset arrival-date field."),
    ("arrival_date_month", 1, "Requested arrival month.", "Dataset arrival-date field."),
    ("arrival_date_week_number", 1, "Requested arrival week.", "Dataset arrival-date field."),
    ("arrival_date_day_of_month", 1, "Requested arrival day.", "Dataset arrival-date field."),
    ("stays_in_weekend_nights", 1, "Requested weekend nights.", "Dataset stay-length field."),
    ("stays_in_week_nights", 1, "Requested weekday nights.", "Dataset stay-length field."),
    ("adults", 1, "Guest count supplied with reservation.", "Dataset guest-count field."),
    ("children", 1, "Requested child count; blanks are a cleaning question.", "Dataset guest-count field."),
    ("babies", 1, "Requested baby count.", "Dataset guest-count field."),
    ("meal", 1, "Requested meal plan.", "Dataset reservation field."),
    ("country", 1, "Guest country recorded with reservation.", "Dataset guest-origin field."),
    ("market_segment", 1, "Booking segment recorded at booking.", "Dataset segment field."),
    ("distribution_channel", 1, "Booking channel known when booking arrives.", "Dataset channel field."),
    ("is_repeated_guest", 1, "Repeat status from booking history.", "Dataset repeat-guest field."),
    ("previous_cancellations", 1, "Historical cancellations available at booking.", "Dataset booking-history field."),
    ("previous_bookings_not_canceled", 1, "Historical completed bookings available at booking.", "Dataset booking-history field."),
    ("reserved_room_type", 1, "Requested room type known at booking.", "Dataset reservation field."),
    ("assigned_room_type", 2, "Assignment can differ and change before arrival.", "Dataset distinguishes reserved and assigned types."),
    ("booking_changes", 2, "Running count can increase after original booking.", "Dataset reservation-change field."),
    ("deposit_type", 1, "Deposit policy attached when booking is made.", "Dataset deposit-policy field."),
    ("agent", 1, "Agent identifier attached to booking; empty means no agent.", "docs/DOMAIN.md."),
    ("company", 1, "Company identifier attached to booking; empty means no company.", "docs/DOMAIN.md."),
    ("days_in_waiting_list", 2, "Duration becomes known after time elapses.", "Dataset waiting-list field."),
    ("customer_type", 1, "Customer type recorded with booking classification.", "Dataset customer-type field."),
    ("adr", 2, "Rate may change through amendments.", "Dataset price field."),
    ("required_car_parking_spaces", 2, "Parking request can change before arrival.", "Dataset request field."),
    ("total_of_special_requests", 2, "Requests can accumulate before arrival.", "Dataset request-count field."),
    ("reservation_status", 3, "Final reservation outcome.", "Dataset outcome-status field."),
    ("reservation_status_date", 3, "Date final status was set.", "Dataset outcome-date field."),
]
audit = pd.DataFrame(audit_rows, columns=["column", "bin", "why", "evidence"])
display(audit)
print(f"Loaded {len(bookings):,} rows and audited {len(audit)} columns.")
print(audit["bin"].value_counts().sort_index().rename("columns"))


,column,bin,why,evidence
0,hotel,1,Selected when reservation is made.,Dataset hotel field.
1,is_canceled,3,Outcome being predicted; known after cancellat...,src/data.py defines the outcome column.
2,lead_time,1,Days between booking and arrival; known at boo...,Dataset booking/arrival field.
3,arrival_date_year,1,Requested arrival year.,Dataset arrival-date field.
4,arrival_date_month,1,Requested arrival month.,Dataset arrival-date field.
5,arrival_date_week_number,1,Requested arrival week.,Dataset arrival-date field.
6,arrival_date_day_of_month,1,Requested arrival day.,Dataset arrival-date field.
7,stays_in_weekend_nights,1,Requested weekend nights.,Dataset stay-length field.
8,stays_in_week_nights,1,Requested weekday nights.,Dataset stay-length field.
9,adults,1,Guest count supplied with reservation.,Dataset guest-count field.


Loaded 119,390 rows and audited 32 columns.
bin
1    23
2     6
3     3
Name: columns, dtype: int64


In [2]:
assert set(audit["column"]) == set(bookings.columns)
assert len(audit) == len(bookings.columns) == 32
assert audit["column"].is_unique
assert audit["bin"].isin([1, 2, 3]).all()
assert set(audit.loc[audit["bin"] == 3, "column"]) == {"is_canceled", "reservation_status", "reservation_status_date"}
print("Audit validation passed: every source column has exactly one valid bin.")


Audit validation passed: every source column has exactly one valid bin.


## Phase 5 handoff

Only Bin 1 columns are eligible for the MVP. Bin 2 columns require an explicit caveat and later comparison. Bin 3 columns are outcomes or post-outcome fields and must never be model predictors. The AUC table in `docs/COLUMNS.md` remains blank until modeling.